# COVID-19 Clonotype Biomarker Reproduction (airr_covid19)

This notebook reproduces cohort-level COVID-associated clonotype analysis with:

- functional clonotype filtering (DNA-based input),
- first-pass batch correction parity with `gene_usage_correction.ipynb`,
- post-correction sample re-normalization,
- whole-cohort Fisher scan,
- depth-aware mode (`test="depth_glm"`),
- concordance checks vs `covid_associated_clonotypes.csv` when available.

The notebook follows a deterministic, staged pipeline and writes benchmark artifacts for runtime tracking.

## 1. Repository Bootstrap and Path Audit (`notebooks/`, `benchmarks/`, `skills/mirpy/`)

Detect the repository root, validate expected folder conventions, and inspect skill path layout (including `mir/_skills`).

In [ ]:
# Configure deterministic runtime context and show key package versions.
from __future__ import annotations

from pathlib import Path
import json
import os
import random
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import polars as pl
import scipy
import statsmodels

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


def find_repo_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()
    for cand in (here, *here.parents):
        if (cand / "pyproject.toml").exists() and (cand / "mir").exists():
            return cand
    raise FileNotFoundError("Could not locate repository root from current notebook path")


repo_root = find_repo_root()
notebooks_dir = repo_root / "notebooks"
benchmarks_dir = repo_root / "benchmarks"
skills_dir = repo_root / "skills" / "mirpy"
mir_skills_dir = repo_root / "mir" / "_skills" / "mirpy"

print(f"repo_root={repo_root}")
print(f"python={sys.version.split()[0]} numpy={np.__version__} pandas={pd.__version__} polars={pl.__version__}")
print(f"scipy={scipy.__version__} statsmodels={statsmodels.__version__}")
print(f"notebooks_dir_exists={notebooks_dir.exists()} benchmarks_dir_exists={benchmarks_dir.exists()}")
print(f"skills/mirpy exists={skills_dir.exists()}")
print(f"mir/_skills/mirpy exists={mir_skills_dir.exists()} is_symlink={mir_skills_dir.is_symlink()}")
if mir_skills_dir.is_symlink():
    print(f"mir/_skills/mirpy -> {os.readlink(mir_skills_dir)}")

## 2. Hugging Face CLI Fetch + Local Cache Management for `isalgo/airr_covid19`

Use deterministic dataset fetch (CLI when available, mirpy helper fallback), then record lightweight checksums for reproducibility.

In [ ]:
# Fetch or refresh airr_covid19 dataset with reproducible local staging.
from mir.utils.notebook_assets import ensure_airr_covid19


def sha256_first_mb(path: Path, mb: int = 1) -> str:
    import hashlib

    h = hashlib.sha256()
    with path.open("rb") as fh:
        h.update(fh.read(mb * 1024 * 1024))
    return h.hexdigest()


dataset_root = repo_root / "notebooks" / "assets" / "large" / "airr_covid19"
dataset_root.mkdir(parents=True, exist_ok=True)

hf_cli = subprocess.run(["bash", "-lc", "command -v hf"], check=False, capture_output=True, text=True)
if hf_cli.returncode == 0:
    hf_cmd = [
        "hf",
        "download",
        "isalgo/airr_covid19",
        "--repo-type",
        "dataset",
        "--local-dir",
        str(dataset_root),
    ]
    print("Running:", " ".join(hf_cmd))
    _ = subprocess.run(hf_cmd, check=False)
else:
    print("hf CLI not found; using mirpy helper fetch")
    dataset_root = ensure_airr_covid19(repo_root=repo_root)

metadata_path = dataset_root / "metadata_trb_min100000.tsv"
if not metadata_path.exists():
    dataset_root = ensure_airr_covid19(repo_root=repo_root)

# Record deterministic checksum snapshots for reproducibility diagnostics.
tracked_files = [
    dataset_root / "metadata.tsv",
    dataset_root / "metadata_trb_min100000.tsv",
]
checksums = {
    p.name: sha256_first_mb(p)
    for p in tracked_files
    if p.exists()
}

print(f"dataset_root={dataset_root}")
print(json.dumps(checksums, indent=2, sort_keys=True))

## 3. Schema Validation and Cohort Assembly

Load metadata and repertoire files, validate required columns, harmonize field types, and assemble the COVID vs healthy cohort.

In [ ]:
# Load metadata, validate schema, and create analysis cohort frame.
meta = pd.read_csv(dataset_root / "metadata_trb_min100000.tsv", sep="\t", dtype={"donor_id": "string"}, low_memory=False)

required_cols = {
    "file_name", "sample_id", "COVID_status", "batch_id", "reads", "locus"
}
missing = sorted(required_cols - set(meta.columns))
if missing:
    raise ValueError(f"Missing required metadata columns: {missing}")

cohort = meta[meta["COVID_status"].isin(["COVID", "healthy"])].copy()
cohort = cohort[cohort["locus"].astype(str).str.upper() == "TRB"].copy()
if "is_bad_reseq" in cohort.columns:
    bad_mask = cohort["is_bad_reseq"].fillna("").astype(str).str.strip().str.lower().isin({"1", "true", "yes"})
    cohort = cohort[~bad_mask].copy()

cohort["reads"] = pd.to_numeric(cohort["reads"], errors="coerce").fillna(0).astype(int)
cohort["batch_id"] = cohort["batch_id"].astype(str)
cohort = cohort.sort_values(["COVID_status", "batch_id", "sample_id"]).reset_index(drop=True)

print(f"cohort_samples={len(cohort)}")
print(cohort["COVID_status"].value_counts(dropna=False))
print(cohort["batch_id"].nunique(), "unique batches")

## 4. Functional-Clonotype Filtering for DNA-Based Counts

Parse TRB repertoires, retain functional/productive clonotypes only, and report pre/post filter depth.

In [ ]:
# Build SampleRepertoire objects with strict functional filtering.
from mir.common.parser import ClonotypeTableParser
from mir.common.repertoire import LocusRepertoire, SampleRepertoire
from mir.common.filter import filter_functional

max_samples = int(os.getenv("MIRPY_COVID_NOTEBOOK_SAMPLES", "240"))
parser = ClonotypeTableParser()

samples_raw: list[SampleRepertoire] = []
pre_post_rows: list[dict[str, int | str]] = []

t_load0 = time.perf_counter()
for _, row in cohort.head(max_samples).iterrows():
    sample_id = str(row["sample_id"])
    path = dataset_root / str(row["file_name"])
    if not path.exists():
        continue

    clones = parser.parse(str(path))
    rep_pre = LocusRepertoire(clonotypes=clones, locus="TRB", repertoire_id=sample_id)
    rep_post = filter_functional(rep_pre)
    if rep_post.clonotype_count == 0:
        continue

    sample = SampleRepertoire(
        loci={"TRB": rep_post},
        sample_id=sample_id,
        sample_metadata={
            "COVID_status": str(row["COVID_status"]),
            "batch_id": str(row["batch_id"]),
            "reads": int(row["reads"]),
            "file_name": str(row["file_name"]),
        },
    )
    samples_raw.append(sample)
    pre_post_rows.append(
        {
            "sample_id": sample_id,
            "covid_status": str(row["COVID_status"]),
            "pre_clonotypes": int(rep_pre.clonotype_count),
            "post_clonotypes": int(rep_post.clonotype_count),
            "pre_duplicates": int(rep_pre.duplicate_count),
            "post_duplicates": int(rep_post.duplicate_count),
        }
    )

load_seconds = time.perf_counter() - t_load0
pre_post_df = pd.DataFrame(pre_post_rows)

print(f"loaded_samples={len(samples_raw)} in {load_seconds:.2f}s")
print(pre_post_df[["pre_clonotypes", "post_clonotypes", "pre_duplicates", "post_duplicates"]].sum())
pre_post_df.head(5)

## 5. Baseline Pipeline Parity with `gene_usage_correction.ipynb`

Apply the same `compute_batch_corrected_gene_usage` + `zscore_to_sigmoid` logic used in the gene-usage correction workflow.

In [ ]:
# Compute first-pass batch-corrected gene usage and derive sample correction factors.
from mir.basic.gene_usage import compute_batch_corrected_gene_usage, zscore_to_sigmoid
from mir.common.repertoire_dataset import RepertoireDataset

samples_map = {s.sample_id: s for s in samples_raw}
metadata_map = {s.sample_id: dict(s.sample_metadata) for s in samples_raw}
dataset = RepertoireDataset(samples=samples_map, metadata=metadata_map)

t_corr0 = time.perf_counter()
usage_corr = compute_batch_corrected_gene_usage(
    dataset,
    batch_field="batch_id",
    scope="vj",
    weighted=True,
    pseudocount=1.0,
)
corr_seconds = time.perf_counter() - t_corr0

if usage_corr.empty:
    raise RuntimeError("Batch-correction table is empty; check cohort assembly and metadata")

usage_corr["corr_weight"] = zscore_to_sigmoid(usage_corr["z"].to_numpy(dtype=float))
sample_weight = usage_corr.groupby("sample_id", dropna=False)["corr_weight"].median().to_dict()

print(f"batch_correction_rows={len(usage_corr)} correction_time_s={corr_seconds:.2f}")
pd.Series(sample_weight).describe()

## 6. Batch Correction and Post-Correction Sample Re-normalization

Rescale per-clonotype duplicate counts with sample-level correction factors, then re-normalize each sample back to original total depth.

In [ ]:
# Build correction-adjusted sample repertoires with per-sample depth renormalization.
from copy import deepcopy

samples_corrected: list[SampleRepertoire] = []
renorm_rows: list[dict[str, float | str | int]] = []

for sample in samples_raw:
    sample_id = sample.sample_id
    weight = float(sample_weight.get(sample_id, 1.0))
    source = sample.get_locus("TRB")
    clones = [deepcopy(c) for c in source.clonotypes]

    original_total = int(sum(int(c.duplicate_count or 0) for c in clones))
    for c in clones:
        base = max(1, int(c.duplicate_count or 1))
        c.duplicate_count = max(1, int(round(base * weight)))

    corrected_total = int(sum(int(c.duplicate_count or 0) for c in clones))
    scale = (original_total / corrected_total) if corrected_total > 0 else 1.0
    for c in clones:
        c.duplicate_count = max(1, int(round(int(c.duplicate_count or 1) * scale)))

    final_total = int(sum(int(c.duplicate_count or 0) for c in clones))
    corrected_rep = LocusRepertoire(clonotypes=clones, locus="TRB", repertoire_id=source.repertoire_id)
    samples_corrected.append(
        SampleRepertoire(
            loci={"TRB": corrected_rep},
            sample_id=sample_id,
            sample_metadata=dict(sample.sample_metadata),
        )
    )
    renorm_rows.append(
        {
            "sample_id": sample_id,
            "correction_weight": weight,
            "original_total": original_total,
            "scaled_total": corrected_total,
            "renormalized_total": final_total,
        }
    )

renorm_df = pd.DataFrame(renorm_rows)
renorm_df.head(5)

## 7. Depth-Aware Statistical Mode Implementation

Configure a selectable depth-aware mode (`depth_glm`) and a baseline mode (`fisher`) on the same target panel.

In [ ]:
# Define shared target panel and association parameters for baseline and depth-aware scans.
from mir.biomarkers.associations import AssociationParams, associate_clonotype_metadata, build_public_clonotype_panel

min_sample_fraction = float(os.getenv("MIRPY_COVID_MIN_SAMPLE_FRACTION", "0.02"))
max_targets = int(os.getenv("MIRPY_COVID_MAX_TARGETS", "1200"))

targets = build_public_clonotype_panel(samples_corrected, locus="TRB", min_sample_fraction=min_sample_fraction)
targets = targets[:max_targets]

params_fisher = AssociationParams(
    match_mode="none",
    count_mode="sample",
    test="fisher",
)
params_depth = AssociationParams(
    match_mode="none",
    count_mode="rearrangement",
    test="depth_glm",
)

print(f"targets={len(targets)} min_sample_fraction={min_sample_fraction}")

## 8. Whole-Cohort Fisher Association Scan for COVID-19 Biomarkers

Run Fisher and depth-aware scans, compute ranked hits, and generate volcano-style summaries.

In [ ]:
# Run whole-cohort association scans for COVID vs healthy labels.
if not targets:
    raise RuntimeError("No targets available for association scan")

t_assoc0 = time.perf_counter()
fisher_result = associate_clonotype_metadata(
    samples_corrected,
    targets,
    metadata_field="COVID_status",
    metadata_value=["COVID", "healthy"],
    params=params_fisher,
)
fisher_seconds = time.perf_counter() - t_assoc0

t_assoc1 = time.perf_counter()
depth_result = associate_clonotype_metadata(
    samples_corrected,
    targets,
    metadata_field="COVID_status",
    metadata_value=["COVID", "healthy"],
    params=params_depth,
)
depth_seconds = time.perf_counter() - t_assoc1

fisher_df = fisher_result.table.to_pandas().sort_values(["q_value", "p_value"], ascending=[True, True]).reset_index(drop=True)
depth_df = depth_result.table.to_pandas().sort_values(["q_value", "p_value"], ascending=[True, True]).reset_index(drop=True)

print(f"fisher_rows={len(fisher_df)} fisher_seconds={fisher_seconds:.2f}")
print(f"depth_rows={len(depth_df)} depth_seconds={depth_seconds:.2f}")
fisher_df.head(10)

In [ ]:
# Plot Fisher and depth-aware volcano summaries.
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")

for frame in (fisher_df, depth_df):
    frame["plot_log2_or"] = pd.to_numeric(frame["log2_odds_ratio"], errors="coerce").fillna(0.0)
    frame["plot_neglog10_q"] = -np.log10(pd.to_numeric(frame["q_value"], errors="coerce").clip(lower=1e-300))

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
for ax, frame, title in [
    (axes[0], fisher_df, "Fisher"),
    (axes[1], depth_df, "Depth-aware GLM"),
]:
    ax.scatter(frame["plot_log2_or"], frame["plot_neglog10_q"], s=10, alpha=0.6, color="#2a9d8f")
    sig = frame[frame["q_value"] < 0.05].head(8)
    for _, row in sig.iterrows():
        ax.text(float(row["plot_log2_or"]), float(row["plot_neglog10_q"]), str(row["junction_aa"]), fontsize=8)
    ax.axhline(-np.log10(0.05), color="#e76f51", linestyle="--", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("log2 odds ratio")
axes[0].set_ylabel("-log10(q)")
plt.tight_layout()
plt.show()

## 9. Reproduction of `covid_associated_clonotypes.csv` and Concordance Checks

Write candidate outputs and compare overlap/rank concordance with the reference file when present.

In [ ]:
# Persist candidate outputs and compute concordance diagnostics.
derived_dir = repo_root / "notebooks" / "assets" / "derived"
derived_dir.mkdir(parents=True, exist_ok=True)

candidate_path = derived_dir / "covid_associated_clonotypes_candidates.csv"
fisher_df.to_csv(candidate_path, index=False)

reference_path = dataset_root / "covid_associated_clonotypes.csv"
concordance = {
    "candidate_path": str(candidate_path),
    "reference_path": str(reference_path),
    "reference_exists": reference_path.exists(),
}

if reference_path.exists():
    ref = pd.read_csv(reference_path)
    if "junction_aa" in ref.columns:
        top_n = 200
        cand_top = fisher_df.head(top_n)["junction_aa"].astype(str)
        ref_set = set(ref["junction_aa"].astype(str))
        overlap = int(cand_top.isin(ref_set).sum())
        concordance.update(
            {
                "top_n": top_n,
                "top_n_overlap": overlap,
                "top_n_jaccard": float(overlap / max(1, len(set(cand_top)) + len(ref_set) - overlap)),
            }
        )
else:
    concordance["note"] = "Reference file is absent in local dataset snapshot"

concordance

## 10. Benchmark Suite (runtime/memory/reproducibility) in `benchmarks/`

Emit machine-readable stage timings and point to benchmark entrypoints in `benchmarks/` and `tests/`.

In [ ]:
# Write benchmark summary JSON for reproducible stage timing diagnostics.
bench_results_dir = repo_root / "benchmarks" / "results"
bench_results_dir.mkdir(parents=True, exist_ok=True)

bench_summary = {
    "seed": SEED,
    "dataset_root": str(dataset_root),
    "sample_count": len(samples_corrected),
    "target_count": len(targets),
    "load_seconds": load_seconds,
    "batch_correction_seconds": corr_seconds,
    "fisher_seconds": fisher_seconds,
    "depth_seconds": depth_seconds,
    "candidate_csv": str(candidate_path),
    "reference_exists": reference_path.exists(),
    "checksums": checksums,
}
summary_path = bench_results_dir / "covid19_associations_notebook_summary.json"
summary_path.write_text(json.dumps(bench_summary, indent=2, sort_keys=True), encoding="utf-8")

print(f"Wrote benchmark summary: {summary_path}")
print("Benchmark scripts:")
print(" - benchmarks/covid19_associations_benchmark.py")
print(" - tests/test_associations_covid19_benchmark.py")

## 11. Notebook + Docs Automation (README and `skills/mirpy/SKILL.md`)

Generate synchronization diagnostics and suggested command snippets for docs/skills consistency updates.

In [ ]:
# Verify docs and skill files reference this workflow and report sync status.
readme_path = repo_root / "README.md"
skill_path = repo_root / "skills" / "mirpy" / "SKILL.md"
docs_mod_path = repo_root / "docs" / "mir.biomarkers.rst"

readme_text = readme_path.read_text(encoding="utf-8")
skill_text = skill_path.read_text(encoding="utf-8")
docs_text = docs_mod_path.read_text(encoding="utf-8")

sync_report = {
    "readme_mentions_notebook": "covid19_biomarkers.ipynb" in readme_text,
    "skill_mentions_notebook": "covid19_biomarkers.ipynb" in skill_text,
    "docs_has_associations_module": "mir.biomarkers.associations module" in docs_text,
    "skill_symlink_ok": mir_skills_dir.is_symlink(),
}

print("Suggested maintenance commands:")
print("  source .venv/bin/activate.fish")
print("  env -C docs make html")
print("  RUN_BENCHMARK=1 pytest -s tests/test_associations_covid19_benchmark.py")
sync_report